<a href="https://colab.research.google.com/github/Muqqadas30/fsdl-my-labs/blob/main/lab05/lab05_testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pytest flake8 black --quiet

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.9/57.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 9.2 MB/s eta 0:00:00


In [2]:
train_raw = datasets.FashionMNIST(root="./data", train=True, download=True)
valid_raw = datasets.FashionMNIST(root="./data", train=False, download=True)

x_train = train_raw.data.float().unsqueeze(1) / 255.0
y_train = train_raw.targets
x_valid = valid_raw.data.float().unsqueeze(1) / 255.0
y_valid = valid_raw.targets

class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]


class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

100%|██████████| 26.4M/26.4M [00:02<00:00, 12.8MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 201kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.75MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 19.3MB/s]


In [3]:
%%writefile model_utils.py
import torch
import torch.nn as nn
import torch.nn.functional as F


def add_five_and_sum(tensor):
    """Add five to every entry of a tensor and return the sum.

    Args:
        tensor: input tensor of any shape.

    Returns:
        The scalar sum after adding 5 to every element.

    Examples:
        >>> import torch
        >>> add_five_and_sum(torch.zeros((2, 2)))
        tensor(20.)
    """
    return (tensor + 5).sum()

Writing model_utils.py


In [4]:
!black model_utils.py
!flake8 model_utils.py

All done! ✨ 🍰 ✨
1 file left unchanged.
model_utils.py:1:1: F401 'torch' imported but unused
model_utils.py:2:1: F401 'torch.nn' imported but unused
model_utils.py:3:1: F401 'torch.nn.functional as F' imported but unused


In [5]:
!pytest --doctest-modules model_utils.py

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0
rootdir: /content
plugins: typeguard-4.5.2, anyio-4.14.2, langsmith-0.10.2
collected 1 item                                                               

model_utils.py .                                                         [100%]

============================== 1 passed in 2.51s ===============================


In [6]:
%%writefile test_model.py
import torch
from model_utils import add_five_and_sum


def test_add_five_and_sum_basic():
    """Sanity check: zeros should sum to 5 per element."""
    result = add_five_and_sum(torch.zeros((2, 2)))
    assert result.item() == 20.0


def test_add_five_and_sum_ones():
    """Adding five to ones should give 6 per element."""
    result = add_five_and_sum(torch.ones((3,)))
    assert result.item() == 18.0

Writing test_model.py


In [7]:
!pytest test_model.py -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: typeguard-4.5.2, anyio-4.14.2, langsmith-0.10.2
collected 2 items                                                              

test_model.py::test_add_five_and_sum_basic PASSED                        [ 50%]
test_model.py::test_add_five_and_sum_ones PASSED                         [100%]

============================== 2 passed in 1.73s ===============================


In [8]:
%%writefile test_cnn_shapes.py
import torch
from model_utils import add_five_and_sum


def test_cnn_output_shape():
    """CNN model ko import karke test karo output shape sahi hai."""
    import sys
    sys.path.append(".")

    # dummy import - actual test yahan model define karke karenge
    class DummyCNN(torch.nn.Module):
        def __init__(self):
            super().__init__()
            self.conv = torch.nn.Conv2d(1, 8, 3, padding=1)
            self.fc = torch.nn.Linear(8 * 28 * 28, 10)

        def forward(self, x):
            x = self.conv(x)
            x = torch.flatten(x, 1)
            return self.fc(x)

    model = DummyCNN()
    batch = torch.randn(4, 1, 28, 28)  # 4 fake images
    output = model(batch)

    assert output.shape == (4, 10), f"Expected (4, 10), got {output.shape}"

Writing test_cnn_shapes.py


In [9]:
!pytest test_cnn_shapes.py -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: typeguard-4.5.2, anyio-4.14.2, langsmith-0.10.2
collected 1 item                                                               

test_cnn_shapes.py::test_cnn_output_shape PASSED                         [100%]

============================== 1 passed in 1.96s ===============================


In [10]:
def test_memorization():
    model = SimpleCNN()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    # Ek chota batch le lo (sirf 8 images)
    xb = x_train[:8]
    yb = y_train[:8]

    initial_loss = None
    for epoch in range(50):
        optimizer.zero_grad()
        outputs = model(xb)
        loss = F.cross_entropy(outputs, yb)
        loss.backward()
        optimizer.step()

        if initial_loss is None:
            initial_loss = loss.item()

    final_loss = loss.item()
    print(f"Initial loss: {initial_loss:.4f} | Final loss: {final_loss:.4f}")

    assert final_loss < 0.5, "Model couldn't memorize a small batch — check for bugs!"
    print("Memorization test PASSED ✅")

test_memorization()

Initial loss: 2.3004 | Final loss: 0.0034
Memorization test PASSED ✅


In [11]:
from torch.profiler import profile, ProfilerActivity

model = SimpleCNN()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

xb = x_train[:64]
yb = y_train[:64]

with profile(activities=[ProfilerActivity.CPU], record_shapes=True) as prof:
    optimizer.zero_grad()
    outputs = model(xb)
    loss = F.cross_entropy(outputs, yb)
    loss.backward()
    optimizer.step()

print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=10))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
autograd::engine::evaluate_function: ConvolutionBack...         0.11%      47.121us        30.59%      12.565ms       6.282ms             2  
                                   ConvolutionBackward0         0.10%      40.613us        30.47%      12.518ms       6.259ms             2  
                             aten::convolution_backward        30.25%      12.426ms        30.37%      12.477ms       6.238ms             2  
                                       aten::max_pool2d         0.24%      99.192us        19.51%       8.013ms       4.007ms             2  
      

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
